<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/LangStudio/RSI_Recent_Detail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import yfinance as yf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

### Remember to load the OptioniVolume.csv file before running the scripts

In [4]:

# ==========================================
# ⚙️ GLOBAL CONFIGURATION (TUNING)
# ==========================================
RSI_LEN = 12            # Lookback period for RSI
RSI_THRESH = 25         # Entry threshold (as defined in your backtest)
LOOKBACK_DAYS = 14      # How far back to search for trigger events
CSV_FILE = "OptionVolume.csv"  # Source for your ticker universe
FALLBACK_SYMBOLS = ["TSLA", "NVDA", "AAPL", "AMD", "MSFT", "BTC-USD"]

# ==========================================
# 📈 PRECISION CALCULATION LOGIC
# ==========================================

def calculate_rsi_precision(series, period=14):
    """Matches Yahoo Finance / Wilder's Smoothing Precision"""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # com=period-1 is the mathematical equivalent to Wilder's Smoothing
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def scan_for_triggers():
    """Main scanning engine for recent RSI events"""
    # 1. Load Symbols
    try:
        df_csv = pd.read_csv(CSV_FILE)
        symbol_col = [c for c in df_csv.columns if 'symbol' in c.lower()][0]
        symbols = df_csv[symbol_col].str.strip().unique().tolist()
        print(f"✅ Loaded {len(symbols)} symbols from {CSV_FILE}")
    except:
        symbols = FALLBACK_SYMBOLS
        print(f"⚠️ {CSV_FILE} not found. Using fallback list: {symbols}")

    report_data = []

    # Calculate dates
    # Fetch 300 extra days to allow RSI math to stabilize (Wilder's Warm-up)
    start_fetch = (datetime.now() - timedelta(days=LOOKBACK_DAYS + 300)).strftime('%Y-%m-%d')
    trigger_cutoff = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).date()

    print(f"🔍 Scanning for RSI {RSI_LEN} < {RSI_THRESH} triggers since {trigger_cutoff}...")

    for symbol in symbols:
        try:
            df = yf.download(symbol, start=start_fetch, progress=False, auto_adjust=True)
            if df.empty or len(df) < RSI_LEN:
                continue

            # Clean columns for multi-index compatibility
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            # Calculate RSI
            df['RSI'] = calculate_rsi_precision(df['Close'], period=RSI_LEN)

            # IDENTIFY THE TRIGGER:
            # RSI crosses below the threshold today, having been above it yesterday
            df['Trigger'] = (df['RSI'] < RSI_THRESH) & (df['RSI'].shift(1) >= RSI_THRESH)

            # Filter for the lookback window
            recent_hits = df[df.index.date >= trigger_cutoff]
            recent_hits = recent_hits[recent_hits['Trigger'] == True]

            for date, row in recent_hits.iterrows():
                report_data.append({
                    "Date": date.strftime('%Y-%m-%d'),
                    "Symbol": symbol,
                    "Price": round(row['Close'], 2),
                    "RSI_Value": round(row['RSI'], 2)
                })
        except Exception as e:
            print(f"❌ Error processing {symbol}: {e}")
            continue

    return report_data

# ==========================================
# 📝 EXECUTION & OUTPUT GENERATION
# ==========================================

if __name__ == "__main__":
    triggers = scan_for_triggers()

    print("\n" + "="*60)
    print(f"🚀 RSI TRIGGER REPORT: LAST {LOOKBACK_DAYS} DAYS")
    print(f"Config: RSI({RSI_LEN}) Threshold < {RSI_THRESH}")
    print("="*60)

    if not triggers:
        print(f"No triggers found in the last {LOOKBACK_DAYS} days.")
    else:
        results_df = pd.DataFrame(triggers)

        # Sort by Date descending (most recent first)
        results_df = results_df.sort_values(by="Date", ascending=False)

        # Display the detailed table
        print(results_df.to_string(index=False))

    print("="*60)

✅ Loaded 150 symbols from OptionVolume.csv
🔍 Scanning for RSI 12 < 25 triggers since 2026-03-05...

🚀 RSI TRIGGER REPORT: LAST 14 DAYS
Config: RSI(12) Threshold < 25
      Date Symbol  Price  RSI_Value
2026-03-19    GDX  82.90      24.99
2026-03-19   BABA 124.90      23.53
2026-03-19     HD 328.21      24.12
2026-03-19      B  38.28      24.16
2026-03-18    UPS  96.84      24.18
2026-03-12    UPS  97.89      24.01
2026-03-12    NKE  54.13      22.89
2026-03-12     BX 102.12      24.29
2026-03-09    UPS  99.94      24.57
2026-03-05    FXI  35.58      22.23


In [6]:
# ==========================================
# ⚙️ GLOBAL CONFIGURATION (MOMENTUM TUNE)
# ==========================================
RSI_LEN = 10            # Shorter length for aggressive momentum (Max Agent style)
RSI_THRESH = 80         # Momentum Ignition threshold (Cross Above)
LOOKBACK_DAYS = 14      # Lookback window for trigger events
SMA_TREND = 200         # Baseline trend filter
CSV_FILE = "OptionVolume.csv"
FALLBACK_SYMBOLS = ["TSLA", "NVDA", "AAPL", "AMD", "MSFT", "COIN", "MSTR"]

# ==========================================
# 📈 PRECISION CALCULATION LOGIC
# ==========================================

def calculate_rsi_precision(series, period=14):
    """Matches Wilder's Smoothing Precision (Yahoo Style)"""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # com=period-1 is equivalent to Wilder's alpha=1/period
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def scan_for_momentum_triggers():
    """Identifies 'Momentum Ignition' (RSI crossing ABOVE threshold)"""
    try:
        df_csv = pd.read_csv(CSV_FILE)
        symbol_col = [c for c in df_csv.columns if 'symbol' in c.lower()][0]
        symbols = df_csv[symbol_col].str.strip().unique().tolist()
    except:
        symbols = FALLBACK_SYMBOLS

    report_data = []

    # Fetch 350 days to ensure SMA 200 and RSI 10 are both stable
    start_fetch = (datetime.now() - timedelta(days=LOOKBACK_DAYS + 350)).strftime('%Y-%m-%d')
    trigger_cutoff = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).date()

    print(f"🚀 Scanning for Momentum Ignition (RSI {RSI_LEN} > {RSI_THRESH}) since {trigger_cutoff}...")

    for symbol in symbols:
        try:
            df = yf.download(symbol, start=start_fetch, progress=False, auto_adjust=True)
            if df.empty or len(df) < SMA_TREND: continue

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            # Indicators
            df['RSI'] = calculate_rsi_precision(df['Close'], period=RSI_LEN)
            df['SMA_200'] = df['Close'].rolling(window=SMA_TREND).mean()

            # MOMENTUM TRIGGER LOGIC:
            # RSI crosses ABOVE the threshold (Power Zone Entry)
            df['Trigger'] = (df['RSI'] > RSI_THRESH) & (df['RSI'].shift(1) <= RSI_THRESH)

            # Trend Labeling
            df['Trend'] = np.where(df['Close'] > df['SMA_200'], "UP", "DOWN")

            recent_hits = df[df.index.date >= trigger_cutoff]
            recent_hits = recent_hits[recent_hits['Trigger'] == True]

            for date, row in recent_hits.iterrows():
                report_data.append({
                    "Date": date.strftime('%Y-%m-%d'),
                    "Symbol": symbol,
                    "Price": f"${row['Close']:.2f}",
                    "RSI": round(row['RSI'], 1),
                    "Trend (SMA200)": row['Trend']
                })
        except Exception:
            continue

    return report_data

# ==========================================
# 📝 EXECUTION & OUTPUT GENERATION
# ==========================================

if __name__ == "__main__":
    triggers = scan_for_momentum_triggers()

    print("\n" + "="*70)
    print(f"🔥 RSI MOMENTUM POWER ZONE REPORT (LAST {LOOKBACK_DAYS} DAYS)")
    print(f"Config: RSI({RSI_LEN}) Crossing ABOVE {RSI_THRESH}")
    print("="*70)

    if not triggers:
        print(f"No momentum breakouts detected in the last {LOOKBACK_DAYS} days.")
    else:
        results_df = pd.DataFrame(triggers)
        results_df = results_df.sort_values(by="Date", ascending=False)

        print(results_df.to_string(index=False))

    print("="*70)

🚀 Scanning for Momentum Ignition (RSI 10 > 80) since 2026-03-05...

🔥 RSI MOMENTUM POWER ZONE REPORT (LAST 14 DAYS)
Config: RSI(10) Crossing ABOVE 80
      Date Symbol   Price  RSI Trend (SMA200)
2026-03-18    USO $121.67 80.2             UP
2026-03-10    USO $105.86 80.8             UP
